# Task Conflict Simulator

This notebook is used for the purposed of developing and validating the task conflict simulator.

## Test with MockLLMClient

Validates the pipeline without making API codes.

In [1]:
import json
from src import llms, utils, task_conflict_generator

def create_mock_responder(prompt: str, system_prompt=None, max_output_tokens: int = 800) -> str:
    """Custom responder for mock testing."""
    if "Write a short document" in prompt:
        return (
            "Paragraph 1: This mock document introduces a controversial workplace policy.\n\n"
            "Paragraph 2: The policy will reduce budgets by 15 percent, even if teams object.\n\n"
            "Paragraph 3: Leaders believe sharper cuts will motivate better performance."
        )
    if "Respond with a JSON object" in prompt:
        return json.dumps({
            "highlighted_sentence": "The policy will reduce budgets by 15 percent, even if teams object.",
            "comment": "This sentence feels too dismissive of the teams' concerns; can we soften it?",
        })
    if "The author would like to keep the doc as it is" in prompt:
        return "I understand the concern, but the firm directive comes straight from leadership and we need to reflect that reality."
    if "The peer still disagrees" in prompt:
        return "We still need to flag that dismissing objections may alienate the staff; please acknowledge the risk."
    return "This is a placeholder response from the mock client."


def preview_mock_entry():
    """Generate a mock dataset entry for preview."""
    mock_llm = llms.MockLLMClient(responder=create_mock_responder)
    return task_conflict_generator.generate_conflict_context(mock_llm, selected_topic=task_conflict_generator._TOPICS[0])

mock_entry = preview_mock_entry()
print(json.dumps(mock_entry.to_dict(), indent=2))

{
  "topic": "a news report on a local incident",
  "document": "This is a placeholder response from the mock client.",
  "highlighted_sentence": "The policy will reduce budgets by 15 percent, even if teams object.",
  "comment_thread": [
    {
      "speaker": "peer",
      "text": "This sentence feels too dismissive of the teams' concerns; can we soften it?"
    },
    {
      "speaker": "author",
      "text": "This is a placeholder response from the mock client."
    },
    {
      "speaker": "peer",
      "text": "This is a placeholder response from the mock client."
    },
    {
      "speaker": "author",
      "text": "This is a placeholder response from the mock client."
    },
    {
      "speaker": "peer",
      "text": "This is a placeholder response from the mock client."
    }
  ]
}


## End to end test with OpenAiClient

Requires OpenAI API key.

In [2]:
llm = llms.OpenAiClient(model='gpt-4o-mini', temperature=0.8)
generated_data = task_conflict_generator.generate_conflict_context_dataset(llm, 1)
print(json.dumps(generated_data[0].to_dict(), indent=2))

{
  "topic": "a news report on a local incident",
  "document": "**News Report: Local Incident Raises Concerns about Community Safety**\n\nOn the evening of March 15th, a troubling incident occurred in the heart of Maplewood, where a group of teenagers was involved in a violent altercation at the town\u2019s central park. Witnesses reported that the confrontation, which escalated quickly, involved approximately ten individuals and resulted in injuries for at least three participants. Local law enforcement arrived promptly on the scene, but the chaos had already drawn a crowd, leaving many residents shaken and questioning the safety of their neighborhood. It is imperative that the community recognizes this event not as a mere isolated incident but as a symptom of deeper issues plaguing our youth and, by extension, our society.\n\nThis alarming event should serve as a wake-up call for our community. It is clear that we can no longer ignore the underlying causes of such violence\u2014issu